# 04 · Phase 2 demo — the learned acoustic model

Phase 1 synthesised speech from 70 averaged phoneme templates. Phase 2 replaces that lookup
table with a trained network, keeping the inputs (phonemes + durations) and the frozen HiFi-GAN
vocoder identical, so the systems stay directly comparable.

| | acoustic model | durations |
| --- | --- | --- |
| **R0** | — (real recording) | — |
| **R1** | real spectrogram | real |
| **A1** | 70 averaged templates | measured (MFA) |
| **A2** | 70 averaged templates | predicted (boosted trees) |
| **D1** | **neural decoder** | measured |
| **D2** | **mel diffusion** | measured |

Run `make dl-prep` first if this is a fresh clone.

In [ ]:
import sys; sys.path.insert(0, "..")
import os; os.chdir("..")

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from src.config import MEL_DIR, OUTPUTS, WAV_DIR, SECONDS_PER_FRAME
from src.utils import read_filelist

SYSTEMS = ["R0", "R1", "A1", "A2", "D1", "D2"]
available = [s for s in SYSTEMS
             if s == "R0" or (OUTPUTS / "audio" / s).exists()
             and any((OUTPUTS / "audio" / s).glob("*.wav"))]
print("audio available for:", available)

## 1 · Listen to the same sentence through every system

In [ ]:
clip_id = read_filelist("test_100")[0]
meta = pd.read_csv("data/metadata_clean.csv", dtype={"id": str}).set_index("id")
print(clip_id, "—", meta.loc[clip_id, "text_clean"])

for s in available:
    d = WAV_DIR if s == "R0" else OUTPUTS / "audio" / s
    p = d / f"{clip_id}.wav"
    if p.exists():
        print(f"\n{s}")
        display(Audio(str(p)))

## 2 · Look at what each acoustic model predicts

The spectrogram is where the difference is visible. Real speech has sharp formant bands and
clear transitions; averaged templates produce flat blocks that change abruptly at phone
boundaries, because every instance of a phone is the same picture.

In [ ]:
import soundfile as sf
from src.audio import wav_to_mel

def mel_of(system, clip_id):
    if system == "R0":
        return np.load(MEL_DIR / f"{clip_id}.npy").astype(np.float32)
    wav, _ = sf.read(OUTPUTS / "audio" / system / f"{clip_id}.wav", dtype="float32")
    return wav_to_mel(wav).numpy()

show = [s for s in available][:4]
fig, axes = plt.subplots(len(show), 1, figsize=(13, 2.4 * len(show)), sharex=True)
axes = np.atleast_1d(axes)
for ax, s in zip(axes, show):
    m = mel_of(s, clip_id)
    ax.imshow(m, origin="lower", aspect="auto", interpolation="nearest", cmap="magma")
    ax.set_ylabel(f"{s}\n80 mels")
axes[-1].set_xlabel("frame (11.61 ms each)")
fig.suptitle(f"{clip_id} — log-mel spectrogram per system", y=0.995)
fig.tight_layout()

## 3 · Say something nobody recorded

Phase 1 could only re-synthesise clips we already had MFA alignments for. `src/dl/g2p.py`
converts arbitrary text to the same phoneme sequence MFA would have produced, so the system can
speak a new sentence.

Pronunciations come from the corpus itself first (what MFA actually used), then M2's OOV
dictionary, then CMUdict.

In [ ]:
from src.dl.g2p import G2P

g2p = G2P()
text = "Phonetic guided speech generation, built for advanced machine learning."
phones = g2p.phonemes(text)
print(" ".join(phones))
print(f"\n{len(phones)} symbols, {phones.count('sil')} silences")
print("ids:", g2p.encode(text)[:20], "...")

To actually hear it you also need predicted durations (M3's duration predictor) and a trained
decoder checkpoint — see `src/dl/synthesize.py --durations neural`.

## 4 · The numbers

In [ ]:
summary = pd.read_csv(OUTPUTS / "results" / "summary.csv")
display(summary)

for name, path in [("duration models", "duration_models.csv"),
                   ("mel cepstral distortion", "mcd.csv"),
                   ("listening study", "listening_study.csv")]:
    p = OUTPUTS / "results" / path
    if p.exists():
        print(f"\n--- {name} ---")
        display(pd.read_csv(p).head(10))

## 5 · Training curve

`outputs/results/train_log.csv` is written by `python -m src.dl.train`. Train and validation
loss falling together is the evidence we are not overfitting.

In [ ]:
log_path = OUTPUTS / "results" / "train_log.csv"
if log_path.exists():
    log = pd.read_csv(log_path)
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.plot(log.step, log.train_loss, label="train", lw=1.2)
    val = log.dropna(subset=["val_loss"]) if "val_loss" in log else log.iloc[0:0]
    if not val.empty:
        ax.plot(val.step, val.val_loss, label="validation", lw=1.6)
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title("D1 training")
else:
    print("no training log yet — run: python -m src.dl.train --system D1")